In [22]:
import pandas as pd
import numpy as np


pd.set_option('display.max_columns', None)


df = pd.read_csv('../data/raw/loan_applications.csv')



df.head()

# apply what I learned in EDA to clean and preprocess the data
df['loan_purpose'] = df['loan_purpose'].str.title().str.strip()
df['employment_type'] = df['employment_type'].str.title().str.strip().str.replace('_', '-').str.replace('Self Employed', 'Self-Employed')


df['fed_funds_rate'] = df['fed_funds_rate'].abs()


df['annual_income'] = df['annual_income'].fillna(df['annual_income'].median())
df['credit_score'] = df['credit_score'].fillna(df['credit_score'].median())
df.head()

,borrower_id,age,employment_type,employment_years,annual_income,credit_score,loan_amount,loan_term_months,loan_purpose,existing_debt,num_open_accounts,num_late_payments,dti_ratio,origination_date,origination_year,origination_quarter,fed_funds_rate,interest_rate_offered
0,BRW02525,42,Salaried,8.5,31904.57,367.0,114000.0,60,Debt Consolidation,16941.85,2,3,0.1062,2022-12-02,2022,Q4,1.374,3.497
1,BRW02922,36,Salaried,7.5,40590.92,360.0,14000.0,12,Auto,5382.22,5,0,0.1326,2023-06-20,2023,Q2,5.278,4.370
2,BRW00889,49,Salaried,17.9,42250.68,355.0,20100.0,36,Debt Consolidation,29697.72,1,0,0.2343,2021-03-03,2021,Q1,0.036,2.500
3,BRW00657,37,Salaried,0.4,82622.40,363.0,124500.0,84,Home Improvement,20650.79,5,0,0.0357,2023-10-26,2023,Q4,5.108,4.989
4,BRW04316,26,Contract,9.3,64728.81,308.0,34100.0,48,Debt Consolidation,230735.46,1,2,0.8912,2021-08-17,2021,Q3,0.127,6.249


Transformation

In [26]:
log_transform_cols = ['annual_income', 'loan_amount', 'existing_debt', 'employment_years']

for col in log_transform_cols:
    # Only transform and drop if the original column still exists
    if col in df.columns:
        df[f'{col}_log'] = np.log1p(df[col])
        df.drop(col, axis=1, inplace=True)

# Same for dti_ratio - only process it if the original column exists
if 'dti_ratio' in df.columns:
    dti_99th = df['dti_ratio'].quantile(0.99)
    df['dti_ratio_capped'] = df['dti_ratio'].clip(upper=dti_99th)
    df['dti_ratio_log'] = np.log1p(df['dti_ratio_capped'])
    df.drop(['dti_ratio', 'dti_ratio_capped'], axis=1, inplace=True)

In [27]:
df.head()

,borrower_id,age,employment_type,credit_score,loan_term_months,loan_purpose,num_open_accounts,num_late_payments,origination_date,origination_year,origination_quarter,fed_funds_rate,interest_rate_offered,annual_income_log,loan_amount_log,existing_debt_log,employment_years_log,dti_ratio_log
0,BRW02525,42,Salaried,367.0,60,Debt Consolidation,2,3,2022-12-02,2022,Q4,1.374,3.497,10.370536,11.643962,9.737601,2.251292,0.100931
1,BRW02922,36,Salaried,360.0,12,Auto,5,0,2023-06-20,2023,Q2,5.278,4.370,10.611324,9.546884,8.591042,2.140066,0.124516
2,BRW00889,49,Salaried,355.0,36,Debt Consolidation,1,0,2021-03-03,2021,Q1,0.036,2.500,10.651399,9.908525,10.298859,2.939162,0.210504
3,BRW00657,37,Salaried,363.0,84,Home Improvement,5,0,2023-10-26,2023,Q4,5.108,4.989,11.322048,11.732069,9.935557,0.336472,0.035078
4,BRW04316,26,Contract,308.0,48,Debt Consolidation,1,2,2021-08-17,2021,Q3,0.127,6.249,11.077977,10.437082,12.349031,2.332144,0.637212



The correlation heatmap showed a 98% correlation between `origination_year` and `fed_funds_rate`. We will drop `origination_year` to avoid multicollinearity. We also drop identifiers and dates as they aren't useful linear predictors.


In [28]:
cols_to_drop = ['borrower_id', 'origination_date', 'origination_year']
df.drop(cols_to_drop, axis=1, inplace=True, errors='ignore')


**apply one hot encoding or get dummies for ml**

In [30]:
categorical_cols = ['employment_type', 'loan_purpose', 'origination_quarter']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
display(df_encoded.head())


,age,credit_score,loan_term_months,num_open_accounts,num_late_payments,fed_funds_rate,interest_rate_offered,annual_income_log,loan_amount_log,existing_debt_log,employment_years_log,dti_ratio_log,employment_type_Retired,employment_type_Salaried,employment_type_Self-Employed,employment_type_Unemployed,loan_purpose_Business,loan_purpose_Debt Consolidation,loan_purpose_Education,loan_purpose_Home Improvement,loan_purpose_Medical,loan_purpose_Vacation,origination_quarter_Q2,origination_quarter_Q3,origination_quarter_Q4
0,42,367.0,60,2,3,1.374,3.497,10.370536,11.643962,9.737601,2.251292,0.100931,False,True,False,False,False,True,False,False,False,False,False,False,True
1,36,360.0,12,5,0,5.278,4.370,10.611324,9.546884,8.591042,2.140066,0.124516,False,True,False,False,False,False,False,False,False,False,True,False,False
2,49,355.0,36,1,0,0.036,2.500,10.651399,9.908525,10.298859,2.939162,0.210504,False,True,False,False,False,True,False,False,False,False,False,False,False
3,37,363.0,84,5,0,5.108,4.989,11.322048,11.732069,9.935557,0.336472,0.035078,False,True,False,False,False,False,False,True,False,False,False,False,True
4,26,308.0,48,1,2,0.127,6.249,11.077977,10.437082,12.349031,2.332144,0.637212,False,False,False,False,False,True,False,False,False,False,False,True,False


# save the file for engineered


In [31]:
df_encoded.to_csv('../data/processed/loan_applications_engineered.csv', index=False)